In [1]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from scipy.stats import randint, uniform, loguniform

from src.metrics import nasa_score
from src.validation import run_cv

In [2]:
df = pd.read_pickle("../data/processed/fd001_basis_c.pkl")
feature_cols_c = [c for c in df.columns if c not in ["unit", "RUL"]]

In [3]:
df.shape

(20631, 100)

In [4]:
len(feature_cols_c)

98

## Tuning: LightGBM und RandomForest

### Ziel
Beide Modelle zeigen mit Default-Parametern eine große Lücke zwischen Train und Val
(Mac mini: LGBM Val 13,24, RF Val 13,46, Train-Werte aus dem Default-Lauf unten).
Frage: Verbessert gezieltes Regularisieren den Val-RMSE, und welches Modell liegt danach vorne?

### Setup
- Basis c (98 Features, geladen aus `fd001_basis_c.pkl`), dieselben 10 GroupKFold-Splits wie in `04`, Mac mini
- `RandomizedSearchCV`, `n_iter`: LGBM 50 / RF 50 (gleicher Selektions-Bias), `random_state = 42`
- `scoring`: RMSE + NASA-Score (`aggregation="mean"` wie in `run_cv`), `refit = "rmse"`
  Begründung: Konsistent mit der Entscheidungsregel vom 22.10. (RMSE ist die Hürde, NASA entscheidet zwischen den Kandidaten). Der NASA-Score bestraft späte Fehler exponentiell und wird dadurch von einzelnen Ausreißer-Triebwerken dominiert. Als Auswahlkriterium für Hyperparameter wäre er zu verrauscht.
- `return_train_score = True` (für die Train/Val-Lücke)
- Parallelisierung: `n_jobs = -1` in der Suche, Modelle mit `n_jobs = 1` (sonst konkurrieren die Prozesse um die Kerne)
- Die Default-Referenz wird in diesem Notebook neu berechnet (gleiche Maschine, gleiche Folds)

### Suchraum LightGBM
| Parameter | Bereich | Verteilung | Default | Warum |
|---|---|---|---|---|
| `num_leaves` | 8–64 | randint | 31 | Weniger Blätter ergeben einfachere Bäume, die weniger Details einzelner Triebwerke abbilden können |
| `min_child_samples` | 10, 20, 40, 80, 160 | Liste (Verdopplung = log-Abstand) | 20 | Größere Blätter mitteln über mehrere Triebwerke statt über aufeinanderfolgende Zyklen eines einzigen |
| `learning_rate` | 0,01–0,2 | loguniform | 0,1 | Kleinere Schritte pro Baum: Das Ensemble lernt vorsichtiger und nimmt weniger Rauschen mit. Log, weil relative Änderungen zählen |
| `n_estimators` | 100–1500 | randint | 100 | Muss zur Lernrate passen: Eine kleine Lernrate braucht mehr Bäume, zu viele Bäume lernen am Ende nur noch die Restfehler (Rauschen) |

Kopplung: `learning_rate` und `n_estimators` hängen zusammen. Gute Kombinationen liegen auf einem diagonalen Streifen.
Prüfung bei der Auswertung: Scatterplot `learning_rate` gegen `n_estimators`, Farbe = RMSE.

### Suchraum RandomForest
| Parameter | Bereich | Verteilung | Default | Warum |
|---|---|---|---|---|
| `max_depth` | None, 8, 12, 16, 20 | Liste | None | Begrenzte Tiefe verhindert, dass Bäume bis zu reinen Blättern wachsen. `None` bleibt drin, damit der Default mitgetestet wird |
| `min_samples_leaf` | 1–50 | randint | 1 | Beim Default 1 kann ein Blatt aus einer einzigen Zeile bestehen, das ist Auswendiglernen. Größere Blätter glätten |
| `max_features` | 0,1–1,0 | uniform | 1,0 | Weniger Features pro Split machen die Bäume unterschiedlicher, dadurch senkt der Mittelwert die Varianz stärker. Nebeneffekt: schneller |
| `n_estimators` | fix 100 | – | 100 | Mehr Bäume senken nur die Varianz bis zu einem Plateau, deshalb fixiert statt getunt |

### Hypothese (vor dem Lauf)
- **H1 LGBM:** Mittlerer RMSE_val > 0,3 besser als Default (13,24) **und** in ≥ 7/10 Folds besser **und** die Train/Val-Lücke schrumpft um ≥ 1,0 (aktuell 6,6)
- **H2 RF:** Mittlerer RMSE_val > 0,3 besser als Default (13,46) **und** in ≥ 7/10 Folds besser **und** die Train/Val-Lücke schrumpft um ≥ 2,0 (aktuell 10,36)
  Die Lücke wird beim RF schwächer gewichtet, weil sein Train-Fehler strukturell klein ist (jeder Baum kennt seine Trainingszeilen fast auswendig).
- **H3:** Nach dem Tuning liegt LGBM weiter vorne, und der Abstand zu RF wächst. Begründung: RF-Defaults gelten als robust, Tuning bringt dort meist wenig. Boosting hat mit `learning_rate`, `n_estimators` und `num_leaves` mehr Hebel. Gegenargument: RF hat die größere Lücke und damit mehr Spielraum.
- **Mechanismus:** Aufeinanderfolgende Zyklen eines Triebwerks sind fast identisch. Kleine Blätter können deshalb einzelne Triebwerke auswendig lernen. Größere Blätter, weniger Komplexität pro Baum und (bei RF) weniger Features pro Split sollten das verhindern: Der Train-RMSE steigt, der Val-RMSE sinkt leicht.
  Schrumpft die Lücke, ohne dass Val besser wird, war Overfitting nicht der Engpass.

### Entscheidungsregel
- H1/H2 gelten als bestätigt, wenn **alle** Teile erfüllt sind.
- Finale Modellwahl (Regel vom 22.10.): Kandidat ist jedes getunte Modell, das seine Hypothese H1/H2 besteht. Bestehen beide, gewinnt das mit dem besseren NASA-Score, weil späte Vorhersagen für die Wartung die gefährliche Richtung sind. Besteht keins, bleibt LGBM mit Default-Parametern.

### Einschränkung
Der beste Such-RMSE ist optimistisch (Selektions-Bias: die beste von `n_iter` Kombinationen, gemessen auf denselben Folds, auf denen sie ausgewählt wurde).
Die ehrliche Zahl liefert erst der Testlauf. Nach dem Testlauf wird nichts mehr getunt.

In [5]:
X = df[feature_cols_c]
y = df["RUL"]
splits = list(GroupKFold(n_splits=10).split(X,y,groups=df["unit"]))

In [6]:
lgbm = run_cv(LGBMRegressor(n_jobs = 1, random_state=42, verbose = -1), X, y, splits)
rf = run_cv(RandomForestRegressor(n_jobs=-1, random_state=42), X, y, splits)

lgbm.to_csv('../data/processed/lgbm_cv.csv', index = False)
rf.to_csv('../data/processed/rf_cv.csv', index = False)

In [7]:
lgbm.drop(columns = "fold").agg("mean")

rmse_train       6.643371
rmse_val        13.241206
nasa_val         3.602103
bias            -0.102583
anteil_spaet     0.461161
max_spaet       54.221874
dtype: float64

In [8]:
rf.drop(columns ="fold").agg("mean")

rmse_train       3.104731
rmse_val        13.459340
nasa_val         3.625276
bias            -0.240784
anteil_spaet     0.392082
max_spaet       53.155000
dtype: float64

In [9]:
RUN_SEARCH = True

nasa_scorer = make_scorer(nasa_score, greater_is_better=False, aggregation = "mean")
scoring = {"rmse": "neg_root_mean_squared_error", "nasa": nasa_scorer}

In [10]:
param_lgbm = {
    "num_leaves": randint(8,65),
    "min_child_samples": [10,20,40,80,160],
    "learning_rate": loguniform(0.01, 0.2),
    "n_estimators": randint(100,1501)
}

param_rf = {
    "max_depth": [None, 8, 12, 16, 20],
    "min_samples_leaf": randint(1, 51),
    "max_features": uniform(0.1,0.9)
}

In [11]:
%%time
if RUN_SEARCH:

    search_lgbm = RandomizedSearchCV(
                    estimator=LGBMRegressor(n_jobs=1, random_state=42, verbose=-1),
                    param_distributions = param_lgbm,
                    n_iter =50,
                    scoring=scoring,
                    refit="rmse",
                    cv=splits,
                    n_jobs=-1,
                    return_train_score= True,
                    random_state=42,
                    )
    search_lgbm.fit(X,y);
    pd.to_pickle(search_lgbm, "../data/processed/search_lgbm.pkl")

search_lgbm = pd.read_pickle("../data/processed/search_lgbm.pkl")    

CPU times: user 9.29 s, sys: 7.82 s, total: 17.1 s
Wall time: 12min 19s


In [12]:
%%time
if RUN_SEARCH:

    search_rf = RandomizedSearchCV(
                    estimator=RandomForestRegressor(n_jobs=1, random_state=42, n_estimators=100),
                    param_distributions = param_rf,
                    n_iter =50,
                    scoring=scoring,
                    refit="rmse",
                    cv=splits,
                    n_jobs=-1,
                    return_train_score= True,
                    random_state=42,
                    )
    search_rf.fit(X,y);
    pd.to_pickle(search_rf, "../data/processed/search_rf.pkl")

search_rf = pd.read_pickle("../data/processed/search_rf.pkl") 

CPU times: user 26.4 s, sys: 7.11 s, total: 33.5 s
Wall time: 40min 53s


In [13]:
res_lgbm = pd.DataFrame(search_lgbm.cv_results_)
display(res_lgbm[["mean_fit_time", "mean_test_rmse", "mean_train_rmse", "split0_test_rmse"]])

res_rf = pd.DataFrame(search_rf.cv_results_)
display(res_rf[["mean_fit_time", "mean_test_rmse", "mean_train_rmse", "split0_test_rmse"]])

,mean_fit_time,mean_test_rmse,mean_train_rmse,split0_test_rmse
0,22.637610,-13.330618,-1.768921,-13.477388
1,2.660859,-13.415681,-5.367438,-13.238270
2,4.333162,-13.583132,-9.558271,-14.147524
3,8.451525,-13.440845,-5.012062,-13.493755
4,13.004379,-13.408943,-1.013739,-13.559384
5,2.396263,-13.059744,-5.577716,-13.430416
6,19.920856,-13.007576,-4.343658,-13.457291
7,10.544703,-13.036234,-7.248175,-13.359538
8,23.760721,-13.147767,-0.760743,-13.570305
9,9.113793,-12.974111,-7.296691,-13.357126


,mean_fit_time,mean_test_rmse,mean_train_rmse,split0_test_rmse
0,64.652955,-13.917436,-10.398193,-14.627747
1,47.798550,-13.277401,-8.294585,-13.822684
2,32.015367,-13.487186,-10.203500,-14.171933
3,8.655440,-13.294866,-7.610741,-14.222166
4,79.291724,-13.355317,-4.432510,-13.960733
5,19.244952,-13.138348,-7.670024,-13.978490
6,29.345476,-13.911685,-11.734206,-14.669017
7,23.513685,-13.486191,-10.473375,-14.309679
8,71.977902,-13.492221,-7.654240,-14.000609
9,31.919043,-13.151098,-6.943479,-13.834297
